In [0]:
import json
from datetime import datetime, timezone
import pandas as ps

# ── Constants ──────────────────────────────────────────────
EH_NAMESPACE    = "eh-ttc-eventhubnamespace"
SAS_KEY_NAME    = "databricks-listener-policy"      # Shared access policy name
SAS_KEY_VALUE   = dbutils.secrets.get(scope="key-vault-secrets", key="eventhub-sas-token")
CONSUMER_GROUP  = "$Default"   # use dedicated consumer group in production

# Build the full connection string from the raw key
BASE_CONN_STRING = (
    f"Endpoint=sb://{EH_NAMESPACE}.servicebus.windows.net/;"
    f"SharedAccessKeyName={SAS_KEY_NAME};"
    f"SharedAccessKey={SAS_KEY_VALUE}"
)


In [0]:
# ── Config builder ─────────────────────────────────────────
def get_event_hub_config(
    eh_name: str,
    start_from: str = "latest",   # "latest" | "beginning" | ISO timestamp string
    consumer_group: str = CONSUMER_GROUP,
    max_events_per_trigger: int = 10_000
) -> dict:
    """
    Build EventHubs config dict for a given hub.
    
    Args:
        eh_name             : Event Hub entity name
        start_from          : "latest", "beginning", or ISO timestamp string
        consumer_group      : Consumer group name
        max_events_per_trigger: Max events per micro-batch
    """
    eh_conn_str = f"{BASE_CONN_STRING};EntityPath={eh_name}"

    if start_from == "latest":
        position = {"offset": "@latest", "seqNo": -1, "enqueuedTime": None, "isInclusive": False}
    elif start_from == "beginning":
        position = {"offset": "-1",      "seqNo": -1, "enqueuedTime": None, "isInclusive": True}
    else:
        # Treat as ISO timestamp e.g. "2024-01-01T00:00:00.000Z"
        position = {"offset": None,      "seqNo": -1, "enqueuedTime": start_from, "isInclusive": True}

    return {
        "eventhubs.connectionString":      sc._jvm.org.apache.spark.eventhubs.EventHubsUtils.encrypt(eh_conn_str),
        "eventhubs.startingPosition":      json.dumps(position),
        "eventhubs.consumerGroup":         consumer_group,
        "eventhubs.maxEventsPerTrigger":   str(max_events_per_trigger),
        "eventhubs.receiverTimeout":       "PT1M",    # ISO 8601 duration — 1 min
        "eventhubs.operationTimeout":      "PT3M",    # ISO 8601 duration — 3 min
    }

In [0]:
from pyspark.sql.functions import col, current_timestamp

# ── Configs for each Event Hub ──────────────────────────────────────────────

eh_config_weather = get_event_hub_config(
    eh_name    = "weatherstreamingeventhub",
    start_from = "latest"
)
eh_config_ttc_service_alerts = get_event_hub_config(
    eh_name    = "ttc_alerts_eventhub",
    start_from = "latest"
)
eh_config_vehicle_positions = get_event_hub_config(
    eh_name    = "ttc_vehicle_positions_eventhub",
    start_from = "latest"
)

# ── Read streams ────────────────────────────────────────────────────────────

df_weather = (
    spark.readStream
    .format("eventhubs")
    .options(**eh_config_weather)
    .load()
    .withColumn("body_str", col("body").cast("string"))
    .withColumn("ingested_at", current_timestamp())
)

df_alerts = (
    spark.readStream
    .format("eventhubs")
    .options(**eh_config_ttc_service_alerts)
    .load()
    .withColumn("body_str", col("body").cast("string"))
    .withColumn("ingested_at", current_timestamp())
)

df_vehicle_positions = (
    spark.readStream
    .format("eventhubs")
    .options(**eh_config_vehicle_positions)
    .load()
    .withColumn("body_str", col("body").cast("string"))
    .withColumn("ingested_at", current_timestamp())
)

In [0]:
CATALOG = "dev"
SCHEMA  = "raw"
CHECKPOINT_BASE = "/Volumes/dev/raw/checkpoints"

query_weather = (
    df_weather.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{CHECKPOINT_BASE}/weather")
    .queryName("weather_stream")
    .toTable(f"{CATALOG}.{SCHEMA}.weather_raw")
)

query_alerts = (
    df_alerts.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{CHECKPOINT_BASE}/ttc_alerts")
    .queryName("ttc_alerts_stream")
    .toTable(f"{CATALOG}.{SCHEMA}.ttc_alerts")
)

query_vehicle_positions = (
    df_vehicle_positions.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{CHECKPOINT_BASE}/vehicle_positions")
    .queryName("vehicle_positions_stream")
    .toTable(f"{CATALOG}.{SCHEMA}.vehicle_positions")
)

print("✓ All 3 streams started concurrently.")
print(f"  • weather_stream          → {CATALOG}.{SCHEMA}.weather")
print(f"  • ttc_alerts_stream       → {CATALOG}.{SCHEMA}.ttc_alerts")
print(f"  • vehicle_positions_stream → {CATALOG}.{SCHEMA}.vehicle_positions")